# Step 3a (seeds): final LambdaRank model over 3 seeds

The final 49-feature model (36 features + 13 popularity, regularised, learning rate 0.05) is retrained with seeds **43** and **44**, to match the attention model's three seeds (42, 43, 44) and report **mean ± sd**.

- **Seed 42 is not retrained.** It is the saved `lambdarank_final_pop.txt` from the main notebook, reloaded and re-scored, and the cell checks it reproduces the logged results.
- **Only the model seed changes.** The within-search shuffle and the metric's tie-breaks stay on seed 42, so all three seeds see identical data and are scored identically.
- **Nothing in `results_step3a.csv` is overwritten.** Outputs go to new files:

| Output | Contents |
|---|---|
| `results_step3a_seeds.csv` | train/val/test NDCG@5, NDCG@10, MRR, purchase hit@10 per seed |
| `results_step3a_seeds_summary.csv` | mean ± sd over the three seeds |
| `predictions_lambdarank_seeds_val.parquet`, `..._test.parquet` | one row per search × product: keys, label, `score_seed42`, `score_seed43`, `score_seed44` |
| `lambdarank_final_pop_seed43.txt`, `..._seed44.txt` | the two new models |

## 0. Setup (same as the main notebook)

In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

!pip -q install lightgbm

Mounted at /content/drive


In [2]:
import glob, json, os, time
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
print('lightgbm', lgb.__version__)

lightgbm 4.6.0


In [3]:
DATA_ROOT  = '/content/drive/MyDrive/bt4222/JDsearch'
STEP1A_DIR = os.path.join(DATA_ROOT, 'step1a_history')
STEP1B_DIR = os.path.join(DATA_ROOT, 'step1b_query_item', 'full')  # train/val/test Parquet
OUT_DIR    = os.path.join(DATA_ROOT, 'step3a_lambdarank')
# Kexun's popularity features: a Parquet file, or a folder of Parquet files (e.g. one per split).
# Leave as None to search DATA_ROOT for a Parquet file with 'pop' in its name.
POP_PATH   = os.path.join(DATA_ROOT, 'step3b_attention', 'step3b_popularity_features.parquet')  # the file the main run found
os.makedirs(OUT_DIR, exist_ok=True)

GROUP_KEY = 'session_id'      # one search = one ranking group
ITEM_KEY  = 'wid'             # product id
LABEL     = 'label'           # 0 none, 1 click, 2 cart, 3 purchase
SPLIT     = 'split'

# Columns that must never be features: ids, the label, the split, and anything positional
NEVER_FEATURES = {GROUP_KEY, ITEM_KEY, LABEL, SPLIT, 'query_id', 'user_id', 'search_id'}
LEAK_PATTERNS  = ['position', 'pos', 'rank', 'order_in_list', 'display_idx']

EVAL_K = [5, 10]

## 1. Load and join the features

In [4]:
import gc
import pyarrow.parquet as pq

STEP1B_ITEM_KEY = 'candidate_id'          # step1b name for the product id
DROP_1B = ['candidate_pos']               # list position leaks the label

def find_split_file(folder, name):
    hits = [f for f in glob.glob(os.path.join(folder, '*.parquet'))
            if name in os.path.basename(f).lower()]
    assert len(hits) == 1, f'expected one {name} file in {folder}, found {hits}'
    return hits[0]

def shrink(d):
    # 64-bit -> 32-bit floats and the smallest integer type; keys stay as they are
    for c in d.columns:
        if pd.api.types.is_float_dtype(d[c]):
            d[c] = d[c].astype('float32')
        elif pd.api.types.is_integer_dtype(d[c]) and c not in (GROUP_KEY, ITEM_KEY):
            d[c] = pd.to_numeric(d[c], downcast='integer')
    return d

def read_cols(path, drop=()):
    keep = [c for c in pq.read_schema(path).names if c not in drop]
    return shrink(pd.read_parquet(path, columns=keep))

def prefixed(d, prefix, keys):
    return d.rename(columns={c: f'{prefix}{c}' for c in d.columns
                             if c not in keys and c not in (LABEL, SPLIT)})

def assert_unique(d, keys, name):
    dup = d.duplicated(keys).sum()
    assert dup == 0, f'{name}: {dup} duplicate rows on {keys}'

# Step 1b: one file per split and the file name is the split
parts = []
for s in ['train', 'val', 'test']:
    d = read_cols(find_split_file(STEP1B_DIR, s), drop=DROP_1B + [SPLIT])
    d = d.rename(columns={STEP1B_ITEM_KEY: ITEM_KEY})
    d[SPLIT] = s
    parts.append(d)
qi = pd.concat(parts, ignore_index=True); del parts, d; gc.collect()
assert_unique(qi, [GROUP_KEY, ITEM_KEY], 'step1b')
n_qi = len(qi)

# Step 1a: label and split are kept only from history_candidate
hc = read_cols(os.path.join(STEP1A_DIR, 'history_candidate_features.parquet'))
assert_unique(hc, [GROUP_KEY, ITEM_KEY], 'history_candidate')
df = prefixed(hc, 'hc__', [GROUP_KEY, ITEM_KEY]); del hc; gc.collect()

df = df.merge(prefixed(qi, 'qi__', [GROUP_KEY, ITEM_KEY]),
              on=[GROUP_KEY, ITEM_KEY], how='inner', suffixes=('', '_qi'))
del qi; gc.collect()
print('rows: step1b', n_qi, '| joined', len(df))
if n_qi != len(df):
    print(f'WARNING: {n_qi - len(df)} step1b rows had no step1a match')

hg = read_cols(os.path.join(STEP1A_DIR, 'history_group_features.parquet'), drop=(LABEL, SPLIT))
assert_unique(hg, [GROUP_KEY, ITEM_KEY], 'history_group')
df = df.merge(prefixed(hg, 'hg__', [GROUP_KEY, ITEM_KEY]), on=[GROUP_KEY, ITEM_KEY], how='left')
del hg; gc.collect()

hu = read_cols(os.path.join(STEP1A_DIR, 'history_user_features.parquet'), drop=(LABEL, SPLIT))
assert_unique(hu, [GROUP_KEY], 'history_user')
df = df.merge(prefixed(hu, 'hu__', [GROUP_KEY]), on=GROUP_KEY, how='left')
del hu; gc.collect()

# Checks: both steps must agree on label and split, and no search spans two splits
assert (df[LABEL] != df[f'{LABEL}_qi']).sum() == 0, 'step1a and step1b labels differ'
df[SPLIT] = df[SPLIT].astype(str).str.lower().replace({'valid': 'val', 'validation': 'val'})
assert (df[SPLIT] != df[f'{SPLIT}_qi']).sum() == 0, 'step1a split differs from step1b file'
df = df.drop(columns=[c for c in df.columns if c.endswith('_qi')])
df[SPLIT] = df[SPLIT].astype('category')
assert (df.groupby(GROUP_KEY, observed=True)[SPLIT].nunique() == 1).all(), 'a search spans splits'

print(f'{len(df):,} rows, {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')
df.groupby(SPLIT, observed=True).agg(searches=(GROUP_KEY, 'nunique'), rows=(ITEM_KEY, 'size'))

rows: step1b 7665446 | joined 7665446
7,665,446 rows, 1.25 GB


,searches,rows
split,,
test,12880,1147673
train,60104,5360982
val,12880,1156791


In [5]:
POP_ITEM_KEYS = ['wid', 'candidate_id']        # whichever name his file uses for the product

def find_pop_files():
    if POP_PATH:
        if os.path.isdir(POP_PATH):
            return sorted(glob.glob(os.path.join(POP_PATH, '*.parquet')))
        return [POP_PATH]
    hits = sorted(f for f in glob.glob(os.path.join(DATA_ROOT, '**', '*.parquet'), recursive=True)
                  if 'pop' in os.path.basename(f).lower() and 'step3a' not in f)
    print('found:', hits)
    assert hits, 'no popularity file found: set POP_PATH in the paths cell'
    return hits

pop_files = find_pop_files()
pop = pd.concat([read_cols(f) for f in pop_files], ignore_index=True)
pop = pop.rename(columns={k: ITEM_KEY for k in POP_ITEM_KEYS if k in pop.columns and k != ITEM_KEY})
pop = pop.drop(columns=[c for c in pop.columns if c in (LABEL, SPLIT) or c in NEVER_FEATURES - {GROUP_KEY, ITEM_KEY}])
pop_keys = [k for k in (GROUP_KEY, ITEM_KEY) if k in pop.columns]   # per search x product, or per product
assert ITEM_KEY in pop_keys, f'no product key in {pop_files}: columns {list(pop.columns)}'
assert_unique(pop, pop_keys, 'popularity')

n0 = len(df)
df = df.merge(prefixed(pop, 'pop__', pop_keys), on=pop_keys, how='left')
assert len(df) == n0, 'popularity merge changed the row count'
pop_cols = [c for c in df.columns if c.startswith('pop__')]
missing = df[pop_cols].isna().all(axis=1)
print(f'joined on {pop_keys}: {len(pop_cols)} popularity columns')
print(pop_cols)
print('rows with no popularity match, by split:', df.loc[missing].groupby(SPLIT, observed=True).size().to_dict())
if len(pop_cols) != 13:
    print('NOTE: expected 13 popularity columns; check the list above against Kexun\'s dictionary')
del pop; gc.collect()

joined on ['session_id', 'wid']: 13 popularity columns
['pop__pop_brand_cart', 'pop__pop_brand_click', 'pop__pop_brand_ord', 'pop__pop_cate4_cart', 'pop__pop_cate4_click', 'pop__pop_cate4_ord', 'pop__pop_product_cart', 'pop__pop_product_click', 'pop__pop_product_ord', 'pop__pop_product_users', 'pop__pop_shop_cart', 'pop__pop_shop_click', 'pop__pop_shop_ord']
rows with no popularity match, by split: {}


2

## 2. Features, ranking groups and metrics

These cells are unchanged from the main notebook, so the rows, the shuffle and the metric are identical.

In [6]:
def is_leak(col):
    base = col.split('__', 1)[-1].lower()
    return any(p == base or base.startswith(p + '_') or base.endswith('_' + p) for p in LEAK_PATTERNS)

numeric = df.select_dtypes(include=[np.number, 'bool']).columns
# popularity columns are listed separately so earlier results stay comparable (section 9b uses them)
POP_FEATURES = [c for c in numeric if c.startswith('pop__')]
FEATURES = [c for c in numeric
            if not c.startswith('pop__')
            and c.split('__', 1)[-1] not in NEVER_FEATURES and c not in NEVER_FEATURES and not is_leak(c)]
dropped_leak = [c for c in numeric if is_leak(c) and not c.startswith('pop__')]
dropped_nonnum = [c for c in df.columns if c not in numeric and c not in NEVER_FEATURES]

GROUPS = {
    'query_item':  [c for c in FEATURES if c.startswith('qi__')],
    'hist_exact':  [c for c in FEATURES if c.startswith('hc__')],
    'hist_group':  [c for c in FEATURES if c.startswith('hg__')],
    'hist_user':   [c for c in FEATURES if c.startswith('hu__')],
    'popularity':  POP_FEATURES,
}
for g, cols in GROUPS.items():
    print(f'{g:11s} {len(cols):3d}  {cols}')
print('\nDropped as possible leakage:', dropped_leak)
print('Dropped as non-numeric (check if any should be encoded):', dropped_nonnum)

df[FEATURES + POP_FEATURES] = df[FEATURES + POP_FEATURES].astype('float32')

query_item   12  ['qi__has_metadata', 'qi__query_name_overlap', 'qi__query_brand_overlap', 'qi__query_cate_overlap', 'qi__bm25_query_name', 'qi__name_length', 'qi__has_brand', 'qi__shop_is_missing', 'qi__cate_depth', 'qi__brand_product_count', 'qi__shop_product_count', 'qi__cate4_product_count']
hist_exact    9  ['hc__prod_n_click', 'hc__prod_n_cart', 'hc__prod_n_follow', 'hc__prod_n_order', 'hc__prod_days_since_click', 'hc__prod_days_since_cart', 'hc__prod_days_since_order', 'hc__prod_days_since_any', 'hc__prod_seen_before']
hist_group   15  ['hg__brand_n_click', 'hg__brand_n_cart', 'hg__brand_n_order', 'hg__brand_days_since_any', 'hg__brand_days_since_order', 'hg__shop_n_click', 'hg__shop_n_cart', 'hg__shop_n_order', 'hg__shop_days_since_any', 'hg__shop_days_since_order', 'hg__cate_n_click', 'hg__cate_n_cart', 'hg__cate_n_order', 'hg__cate_days_since_any', 'hg__cate_days_since_order']
hist_user    17  ['hu__n_click_7d', 'hu__n_cart_90d', 'hu__n_follow_90d', 'hu__n_order_rows_365d', '

In [7]:
has_pos = df.groupby(GROUP_KEY)[LABEL].transform('max') > 0
print('searches with no engagement, by split:')
print(df.loc[~has_pos].groupby(SPLIT, observed=True)[GROUP_KEY].nunique().to_dict())

shuffle_rng = np.random.default_rng(SEED)
data = {}
for s in ['train', 'val', 'test']:
    d = df[(df[SPLIT] == s) & has_pos]
    # random order inside each search, then group rows of a search together
    d = (d.assign(_shuffle=shuffle_rng.random(len(d)))
          .sort_values([GROUP_KEY, '_shuffle'])
          .drop(columns='_shuffle')
          .reset_index(drop=True))
    data[s] = d
    print(f'{s:5s} searches {d[GROUP_KEY].nunique():>7,}  rows {len(d):>9,}')

def group_sizes(d):
    return d.groupby(GROUP_KEY, sort=False).size().to_numpy()

del df, has_pos, d; gc.collect()

searches with no engagement, by split:
{}
train searches  60,104  rows 5,360,982
val   searches  12,880  rows 1,156,791
test  searches  12,880  rows 1,147,673


0

In [8]:
def rank_metrics(d, score, ks=EVAL_K, seed=SEED):
    rng = np.random.default_rng(seed)
    t = pd.DataFrame({'g': d[GROUP_KEY].to_numpy(), 'y': d[LABEL].to_numpy(),
                      's': np.asarray(score, dtype='float64'), 'tie': rng.random(len(d))})
    t = t.sort_values(['g', 's', 'tie'], ascending=[True, False, True])
    t['r'] = t.groupby('g').cumcount() + 1
    t['gain'] = 2.0 ** t['y'] - 1
    t['disc'] = 1 / np.log2(t['r'] + 1)

    ideal = t.sort_values(['g', 'y'], ascending=[True, False]).copy()
    ideal['r'] = ideal.groupby('g').cumcount() + 1
    ideal['disc'] = 1 / np.log2(ideal['r'] + 1)

    out = {}
    for k in ks:
        dcg  = (t['gain'] * t['disc']).where(t['r'] <= k, 0).groupby(t['g']).sum()
        idcg = (ideal['gain'] * ideal['disc']).where(ideal['r'] <= k, 0).groupby(ideal['g']).sum()
        out[f'ndcg@{k}'] = (dcg / idcg).mean()
    first = t[t['y'] > 0].groupby('g')['r'].min()
    out['mrr'] = (1 / first).mean()
    buy = t[t['y'] == 3]
    if len(buy):
        out['purchase_hit@10'] = (buy.groupby('g')['r'].min() <= 10).mean()
        out['n_purchase_searches'] = buy['g'].nunique()
    out['n_searches'] = t['g'].nunique()
    return out

results = []
def log_result(model, feature_set, scores_by_split, notes=''):
    for s, sc in scores_by_split.items():
        results.append({'model': model, 'features': feature_set, 'split': s, **rank_metrics(data[s], sc), 'notes': notes})
    # save after every model, so a Colab disconnect never loses finished results
    pd.DataFrame(results).to_csv(os.path.join(OUT_DIR, 'results_step3a.csv'), index=False)

In [9]:
BASE_PARAMS = dict(
    objective='lambdarank',
    metric='ndcg',
    eval_at=[10],                  # early stopping target
    label_gain=[0, 1, 3, 7],       # gain 2^label - 1, same as our metric
    learning_rate=0.05,
    num_leaves=63,
    min_data_in_leaf=50,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    lambdarank_truncation_level=20,
    seed=SEED,
    verbose=-1,
)
FAST = dict(learning_rate=0.1)     # for ablations and the sweep

def train_ranker(features, params=None, rounds=2000, log_every=50):
    p = {**BASE_PARAMS, **(params or {})}
    dtr = lgb.Dataset(data['train'][features], data['train'][LABEL], group=group_sizes(data['train']))
    dva = lgb.Dataset(data['val'][features],   data['val'][LABEL],   group=group_sizes(data['val']), reference=dtr)
    t0 = time.time()
    m = lgb.train(p, dtr, num_boost_round=rounds, valid_sets=[dva], valid_names=['val'],
                  callbacks=[lgb.early_stopping(50, first_metric_only=True, verbose=False),
                             lgb.log_evaluation(log_every)])
    print(f'best iteration {m.best_iteration}, {(time.time() - t0) / 60:.1f} min, {len(features)} features')
    if m.best_iteration < 10:
        print('NOTE: stopped very early. Fine for a weak feature set; if every model does this, check the shuffle in section 4.')
    return m

def predict_all(m, features):
    return {s: m.predict(data[s][features], num_iteration=m.best_iteration) for s in data}

## 3. Seed 42: reload the saved final model

The saved file already stops at the best round, so `predict` uses every tree in it. The check compares against the `lambdarank_final_pop` rows in `results_step3a.csv`; a mismatch means the data or features differ from the main run.

In [10]:
SEEDS = [42, 43, 44]
POP_FINAL_FEATS = GROUPS['query_item'] + GROUPS['hist_exact'] + GROUPS['hist_group'] + GROUPS['popularity']
FINAL_PARAMS = dict(num_leaves=63, min_data_in_leaf=300, lambda_l2=10.0, feature_fraction=0.6, learning_rate=0.05)
assert len(POP_FINAL_FEATS) == 49, len(POP_FINAL_FEATS)

m42 = lgb.Booster(model_file=os.path.join(OUT_DIR, 'lambdarank_final_pop.txt'))
assert m42.feature_name() == POP_FINAL_FEATS, 'saved model was trained on a different feature list or order'
scores = {42: {s: m42.predict(data[s][POP_FINAL_FEATS]) for s in data}}
best_iter = {42: m42.num_trees()}

logged = (pd.read_csv(os.path.join(OUT_DIR, 'results_step3a.csv'))
            .query("model == 'lambdarank_final_pop'").set_index('split')['ndcg@10'])
for s in ['train', 'val', 'test']:
    ours = rank_metrics(data[s], scores[42][s])['ndcg@10']
    print(f'seed 42 {s:5s} ndcg@10 {ours:.4f} | logged {logged[s]:.4f}')
    assert abs(ours - logged[s]) < 1e-4, f'seed 42 does not reproduce on {s}'
print('seed 42 reproduces the main run; trees:', best_iter[42])

seed 42 train ndcg@10 0.4435 | logged 0.4435
seed 42 val   ndcg@10 0.4072 | logged 0.4072
seed 42 test  ndcg@10 0.4059 | logged 0.4059
seed 42 reproduces the main run; trees: 495


## 4. Seeds 43 and 44: retrain with the same settings

Same features, settings, early stopping (validation NDCG@10, patience 50) and round limit as the seed-42 model; only LightGBM's `seed` changes, which sets the random row bagging (80%) and feature sampling (60%). Each model is saved as soon as it finishes, so a disconnect loses at most one seed. About 20 minutes per seed.

In [11]:
for sd in [43, 44]:
    print(f'\n== seed {sd}')
    m = train_ranker(POP_FINAL_FEATS, {**FINAL_PARAMS, 'seed': sd}, rounds=2000, log_every=100)
    m.save_model(os.path.join(OUT_DIR, f'lambdarank_final_pop_seed{sd}.txt'), num_iteration=m.best_iteration)
    scores[sd] = predict_all(m, POP_FINAL_FEATS)
    best_iter[sd] = m.best_iteration
    del m; gc.collect()


== seed 43
[100]	val's ndcg@10: 0.399466
[200]	val's ndcg@10: 0.4033
[300]	val's ndcg@10: 0.405148
[400]	val's ndcg@10: 0.406331
[500]	val's ndcg@10: 0.406938
best iteration 476, 17.1 min, 49 features

== seed 44
[100]	val's ndcg@10: 0.399998
[200]	val's ndcg@10: 0.40416
[300]	val's ndcg@10: 0.40515
[400]	val's ndcg@10: 0.406325
best iteration 402, 14.7 min, 49 features


## 5. Results per seed and mean ± sd

The sd is the sample standard deviation over the three seeds (pandas default, `ddof=1`).

In [12]:
METRICS = ['ndcg@5', 'ndcg@10', 'mrr', 'purchase_hit@10']
rows = []
for sd in SEEDS:
    for s in ['train', 'val', 'test']:
        rows.append({'model': 'lambdarank_final_pop', 'seed': sd, 'split': s, 'best_iter': best_iter[sd],
                     **rank_metrics(data[s], scores[sd][s])})
per_seed = pd.DataFrame(rows)
per_seed.to_csv(os.path.join(OUT_DIR, 'results_step3a_seeds.csv'), index=False)

g = per_seed.groupby('split')[METRICS]
mean, sdev = g.mean(), g.std()
summary = pd.DataFrame({c: [f'{m:.4f} ± {v:.4f}' for m, v in zip(mean[c], sdev[c])] for c in METRICS}, index=mean.index)
summary = summary.reindex(['train', 'val', 'test'])
summary['seeds'] = len(SEEDS)
summary.to_csv(os.path.join(OUT_DIR, 'results_step3a_seeds_summary.csv'))

display(per_seed.pivot_table(index='seed', columns='split', values='ndcg@10')[['train', 'val', 'test']].round(4)
        .assign(best_iter=pd.Series(best_iter)))
summary

split,train,val,test,best_iter
seed,,,,
42,0.4435,0.4072,0.4059,495
43,0.4415,0.4070,0.4069,476
44,0.4363,0.4067,0.4058,402


,ndcg@5,ndcg@10,mrr,purchase_hit@10,seeds
split,,,,,
train,0.3946 ± 0.0040,0.4404 ± 0.0037,0.4523 ± 0.0042,0.6760 ± 0.0022,3
val,0.3601 ± 0.0004,0.4070 ± 0.0003,0.4158 ± 0.0004,0.6557 ± 0.0023,3
test,0.3593 ± 0.0007,0.4062 ± 0.0006,0.4099 ± 0.0007,0.6389 ± 0.0013,3


## 6. Save validation and test predictions for every seed

One file per split, keyed on `session_id` + `wid`, so they join to the attention model's predictions for paired comparisons.

In [13]:
for s in ['val', 'test']:
    out = data[s][[GROUP_KEY, ITEM_KEY, LABEL]].copy()
    for sd in SEEDS:
        out[f'score_seed{sd}'] = scores[sd][s].astype('float32')
    out['score_mean'] = out[[f'score_seed{sd}' for sd in SEEDS]].mean(axis=1)
    path = os.path.join(OUT_DIR, f'predictions_lambdarank_seeds_{s}.parquet')
    out.to_parquet(path, index=False)
    print(f'{s:4s} {len(out):,} rows -> {path}')

# the seed-42 column must equal the test scores already saved for step 4
old = pd.read_parquet(os.path.join(OUT_DIR, 'test_predictions_lambdarank.parquet'), columns=[GROUP_KEY, ITEM_KEY, 'score_lambdarank_pop'])
chk = out.merge(old, on=[GROUP_KEY, ITEM_KEY])
assert len(chk) == len(out) and np.allclose(chk['score_seed42'], chk['score_lambdarank_pop'], atol=1e-5)
print('seed-42 test scores match test_predictions_lambdarank.parquet')

val  1,156,791 rows -> /content/drive/MyDrive/bt4222/JDsearch/step3a_lambdarank/predictions_lambdarank_seeds_val.parquet
test 1,147,673 rows -> /content/drive/MyDrive/bt4222/JDsearch/step3a_lambdarank/predictions_lambdarank_seeds_test.parquet
seed-42 test scores match test_predictions_lambdarank.parquet


## 7. Summary

- **Final model (49 features, regularised, learning rate 0.05) over seeds 42, 43, 44:** test NDCG@10 **0.4062 ± 0.0006**, validation 0.4070 ± 0.0003, train 0.4404 ± 0.0037.
- **Other test metrics:** MRR 0.4099 ± 0.0007, purchase hit@10 0.6389 ± 0.0013, NDCG@5 0.3593 ± 0.0007.
- **Seed 42 reproduces the main run exactly**, so the three seeds differ only in LightGBM's random row and feature sampling; the data, shuffle and metric tie-breaks are identical.
- **The spread across seeds is small:** a test sd of 0.0006 is about 0.15% of the mean, so the result does not depend on a lucky seed.
- **Against the attention model on the same 49 features** (0.3991 ± 0.0010, three seeds, Section 2.3): LambdaRank is higher by about 0.007 test NDCG@10, several times either model's seed sd.